# Modelagem preditiva
_Machine Learning_

---

## Sumário

1. **Importação de bibliotecas**
2. **Carregamento da base**
    - 2.1. Carregamento dos dataframes
    - 2.2. Extração de amostra dos dataframes
3. **Preparação dos dados**
    - 3.1. Exibição dos metadados
    - 3.2. Análise de cardinalidade
    - 3.3. Análise de variáveis não numéricas
    - 3.4. Análise de variáveis binárias
    - 3.5. Selecionando variáveis não aplicáveis à modelagem
    - 3.6. Segmentação das bases **train** e **test**
    - 3.7. Transformação das features das bases **train** e **test**

## 1. Importação de bibliotecas

In [1]:
# Importação de pacotes e definição de parâmetros globais

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.dataset as ds
import warnings
import gc
import os
import time
import optuna
import joblib
import random

from sklearn.model_selection import train_test_split
from pathlib import Path

d:\PROJETOS\analise_e_classificacao_de_churn_clientes_de_banco\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configurações para exibição de dados no Jupyter Notebook

# Configurar opção para exibir todas as linhas do Dataframe
pd.set_option('display.max_rows', None)

# Configurar para exibir o conteúdo completo das colunas
pd.set_option('display.max_colwidth', None)

# Configurar a supressão de mensagens de aviso durante a execução
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos do Seaborn
sns.set_style('whitegrid')

## 2. Carregamento da base

In [3]:
# Efetuando a limpeza da memória antes do carregamento dos dados

print(f'\nQuantidade de objetos removidos da memória: {gc.collect()}')


Quantidade de objetos removidos da memória: 20


In [4]:
# Caminho para o diretório de dados
DATA_DIR = Path('../data/refined')

# Carregando os dados
try:
    dataset = ds.dataset(DATA_DIR, format='parquet')
    table = dataset.to_table()
    df = table.to_pandas()
except Exception as e:
    print(f'Erro ao carregar os dados: {e}')

In [5]:
print('VOLUMETRIA')
print(f'Quantidade de linhas (registros):  {df.shape[0]:,}')
print(f'Quantidade de colunas (variáveis): {df.shape[1]:,}')

VOLUMETRIA
Quantidade de linhas (registros):  10,000
Quantidade de colunas (variáveis): 34


In [6]:
df.head(10)

,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,...,relationship_strength,engagement_index,inactivity_risk_flag,high_value_customer,high_balance_inactive,high_value_low_products,senior_high_balance,inactive_high_salary,inactive_multi_product,inactive_long_term_customer
0,15565714,601,France,Male,47,1,64430.06,2,0,1,...,7.5,4,0,0,0,0,0,0,0,0
1,15565806,532,France,Male,38,9,0.00,2,0,0,...,8.5,2,0,0,0,0,0,0,1,1
2,15565879,845,France,Female,28,9,0.00,2,1,1,...,11.5,5,0,0,0,0,0,0,0,0
3,15565891,709,France,Male,39,8,0.00,2,1,0,...,8.0,3,0,0,0,0,0,0,1,1
4,15565996,653,France,Male,44,8,0.00,2,1,1,...,11.0,5,0,0,0,0,0,0,0,0
5,15566111,596,France,Male,39,9,0.00,1,1,0,...,6.5,2,0,0,0,0,0,0,0,1
6,15566139,526,France,Female,37,5,53573.18,1,1,0,...,4.5,2,1,0,0,0,0,0,0,0
7,15566251,618,France,Female,37,5,96652.86,1,1,0,...,4.5,2,1,0,0,0,0,0,0,0
8,15566269,787,France,Male,25,5,0.00,2,1,0,...,6.5,3,0,0,0,0,0,0,1,0
9,15566295,761,France,Female,33,6,138053.79,2,1,0,...,7.0,3,1,1,1,0,0,0,1,0


## 3. Preparação dos dados

### 3.1. Exibição dos metadados

In [7]:
# Função para geração de um dataframe de metadados

def generate_metadata(dataframe):
    '''
    Gera um DataFrame contendo metadados das colunas do DataFrame fornecido.

    :param dataframe: DataFrame
        DataFrame para o qual os metadados serão gerados.
    :return: DataFrame
        DataFrame contendo os metadados.
    '''
    metadata = pd.DataFrame({
        'Variável': dataframe.columns,
        'Tipo': dataframe.dtypes,
        'Qtde de nulos': dataframe.isnull().sum(),
        '% de nulos': round((dataframe.isnull().sum() / len(dataframe)) * 100, 2),
        'Cardinalidade': dataframe.nunique(),
    }).sort_values(by='Qtde de nulos', ascending=False).reset_index(drop=True)

    return metadata

In [8]:
generate_metadata(df)

,Variável,Tipo,Qtde de nulos,% de nulos,Cardinalidade
0,CustomerId,int32,0,0.0,10000
1,CreditScore,int32,0,0.0,460
2,Geography,str,0,0.0,3
3,Gender,str,0,0.0,2
4,Age,int32,0,0.0,70
5,Tenure,int32,0,0.0,11
6,Balance,float64,0,0.0,6382
7,NumOfProducts,int32,0,0.0,4
8,HasCrCard,int32,0,0.0,2
9,IsActiveMember,int32,0,0.0,2


### 3.2. Análise de cardinalidade

In [ ]:
# Listando todas as variáveis com cardinalidade inferior a 2

cols_low_cardinality = [col for col in df.columns if df[col].nunique() < 2]
print(cols_low_cardinality)

Variáveis com cardinalidade inferior a 2: ['processing_date']


### 3.3. Análise de variáveis não numéricas

In [12]:
# Listando todas as variáveis string, object e category

cols_text = [col for col in df.columns if df[col].dtype in ['string', 'object', 'category']]
print(cols_text)

['Geography', 'Gender', 'age_bucket', 'tenure_bucket', 'processing_date']


In [13]:
df[cols_text].head(10)

,Geography,Gender,age_bucket,tenure_bucket,processing_date
0,France,Male,Adult,New customer,2026-04-01
1,France,Male,Adult,Long term,2026-04-01
2,France,Female,Young,Long term,2026-04-01
3,France,Male,Adult,Long term,2026-04-01
4,France,Male,Adult,Long term,2026-04-01
5,France,Male,Adult,Long term,2026-04-01
6,France,Female,Adult,Mid tenure,2026-04-01
7,France,Female,Adult,Mid tenure,2026-04-01
8,France,Male,Young,Mid tenure,2026-04-01
9,France,Female,Adult,Mid tenure,2026-04-01


### 3.4. Análise de variáveis binárias

In [17]:
# Listando todas as variáveis binárias
binary_cols = [col for col in df.columns
    if set(df[col].dropna().unique()) == {0, 1}]
print(binary_cols)

# Transformando as variáveis binárias em categóricas
df[binary_cols] = df[binary_cols].astype('category')

['HasCrCard', 'IsActiveMember', 'is_high_credit_score', 'is_senior_new_customer', 'is_zero_balance', 'is_multi_product_customer', 'is_active_with_card', 'Exited', 'inactivity_risk_flag', 'high_value_customer', 'high_balance_inactive', 'high_value_low_products', 'senior_high_balance', 'inactive_high_salary', 'inactive_multi_product', 'inactive_long_term_customer']


In [18]:
df[binary_cols].head(10)

,HasCrCard,IsActiveMember,is_high_credit_score,is_senior_new_customer,is_zero_balance,is_multi_product_customer,is_active_with_card,Exited,inactivity_risk_flag,high_value_customer,high_balance_inactive,high_value_low_products,senior_high_balance,inactive_high_salary,inactive_multi_product,inactive_long_term_customer
0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,1
2,1,1,1,0,1,1,1,0,0,0,0,0,0,0,0,0
3,1,0,0,0,1,1,0,0,0,0,0,0,0,0,1,1
4,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0
5,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1
6,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
7,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0
8,1,0,1,0,1,1,0,0,0,0,0,0,0,0,1,0
9,1,0,1,0,0,1,0,0,1,1,1,0,0,0,1,0


### 3.5. Selecionando variáveis não aplicáveis à modelagem

In [19]:
# Separando as variáveis não aplicáveis para modelagem preditiva

vars_to_exclude = ['CustomerId', 'processing_date']
df.drop(columns=vars_to_exclude, inplace=True)

In [20]:
print('VOLUMETRIA')
print(f'Quantidade de linhas (registros):  {df.shape[0]:,}')
print(f'Quantidade de colunas (variáveis): {df.shape[1]:,}')

VOLUMETRIA
Quantidade de linhas (registros):  10,000
Quantidade de colunas (variáveis): 32
